# Financial analysis with `%cash_on`

A small analysis pipeline over ten years of daily prices for eight tickers:
load, compute rolling features, summarise, loop over tickers. Run it top to
bottom once, then follow the **Try this** notes. Each one changes one thing and
shows that only the work that depends on it runs again.

**Read the badge, not the clock.** cash prints a badge under every cell it runs.
Open it to see one row per statement: **CACHED** (green) means cash restored the
value instead of running the code, **EXECUTED** (ochre) means the code ran.
Printed output is no evidence either way: a restored statement replays what it
printed.

In [ ]:
import cash
%cash_on

## 0. The data

The next cell writes `large_financial_data.csv` next to this notebook (about
20,000 rows from a fixed seed) if it is not there yet.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

data_file = Path("large_financial_data.csv")
if not data_file.exists():
    rng = np.random.default_rng(42)
    dates = pd.bdate_range("2015-01-01", periods=2520)
    frames = []
    for ticker in ["AAPL", "AMZN", "GOOGL", "META", "MSFT", "NFLX", "NVDA", "TSLA"]:
        close = 100 * np.exp(np.cumsum(rng.normal(0.0004, 0.02, len(dates))))
        volume = rng.integers(1_000_000, 50_000_000, len(dates))
        frames.append(pd.DataFrame({"Date": dates, "Ticker": ticker, "Close": close.round(2), "Volume": volume}))
    pd.concat(frames).sort_values(["Date", "Ticker"]).to_csv(data_file, index=False)

## 1. Load and sort

Each step binds a new value to `df` instead of changing it in place. That is
what lets cash cache it: a statement that mutates an existing object has no
result to store.

In [ ]:
df = pd.read_csv(data_file, parse_dates=["Date"])
df = df.sort_values(["Ticker", "Date"])
df.head()

## 2. Rolling features, one statement each

Two slow rolling-window features, each in its own statement, then one cheap
statement that joins them.

**Try this:** change the window in the `vol_adj` line from `20` to `30` and run
the cell again. `vol_adj` and `features` say EXECUTED; `wsma` says CACHED,
because it does not read anything you changed.

In [ ]:
def vol_adjusted_mean(window):
    return np.mean(window) / (np.std(window) + 1e-6)


def weighted_mean(window):
    weights = np.arange(1, len(window) + 1)
    return np.sum(window * weights) / np.sum(weights)


vol_adj = df.groupby("Ticker")["Close"].transform(lambda s: s.rolling(20).apply(vol_adjusted_mean, raw=True))
wsma = df.groupby("Ticker")["Close"].transform(lambda s: s.rolling(50).apply(weighted_mean, raw=True))
features = df.assign(VolAdj=vol_adj, WSMA=wsma)
features.tail()

## 3. A third feature in its own cell

In [ ]:
def rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    return 100 - 100 / (1 + gain / loss)


with_rsi = features.assign(RSI=features.groupby("Ticker")["Close"].transform(rsi))
with_rsi.tail()

## 4. Summary per ticker

In [ ]:
summary = with_rsi.groupby("Ticker").agg(
    mean_close=("Close", "mean"),
    std_close=("Close", "std"),
    total_volume=("Volume", "sum"),
    mean_rsi=("RSI", "mean"),
    last_wsma=("WSMA", "last"),
)
summary

## 5. A loop, cached one iteration at a time

cash caches each iteration of a loop on its own. `slow_stats` sleeps a little
to stand in for real work.

**Try this:** add `"MSFT"` to the list and run the cell again. The three
tickers you already had say CACHED; only the new one runs.

In [ ]:
def slow_stats(prices):
    time.sleep(0.5)  # stands in for real work
    return {"mean": prices.mean(), "std": prices.std(), "last": prices.iloc[-1]}


ticker_stats = {}
for ticker in ["TSLA", "GOOGL", "AAPL"]:
    ticker_stats[ticker] = slow_stats(with_rsi.loc[with_rsi["Ticker"] == ticker, "Close"])

pd.DataFrame(ticker_stats).T

## 6. Branches

Only the branch that runs is cached. The other branch gets no row in the badge.

In [ ]:
if len(df) > 10_000:
    sample = df.sample(n=5_000, random_state=42)
    report_type = "sampled"
else:
    sample = df
    report_type = "full"

print(f"Report type: {report_type}, sample size: {len(sample):,}")

## 7. Where a value came from

`%cash_provenance` shows how a variable was computed and what it depends on.

In [ ]:
%cash_provenance summary --graph

In [ ]:
%cash_stats